<a href="https://colab.research.google.com/github/eshwar-7419/cads/blob/main/CARC_IDS_Phase2B_Continual_Learning_Benchmark_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2B — Continual Learning Benchmark
## CARC-IDS/IPS

This notebook implements the first actual continual-learning experiment after the corrected Phase-2A drift analysis.

### Experimental stream

The stream is based on the evidence already established from the UNSW-NB15 `temporal` training split:

- **E1 — Initial attack regime:** Segment 3
- **E2 — High-attack regime:** Segments 4–6
- **E3 — Attack-family transition:** Segment 7
- **E4 — Generic-dominant regime:** Segment 8
- **E5 — Stable late regime:** Segments 9–10

The original supplied temporal test set is held out and is NOT used for constructing the continual-learning experiences.

### Models compared

1. **Static LightGBM**
2. **Naive continual learning**
3. **Replay continual learning**
4. **EWC-style continual learning**

The purpose of this phase is NOT yet to implement the final resource-aware controller.

It establishes whether continual adaptation itself is useful and how much catastrophic forgetting / computational cost occurs.

### Important methodological choices

- Original UNSW-NB15 features are used.
- `label`, `attack_cat`, and identifier columns are excluded.
- Each experience is split chronologically into adaptation and evaluation portions.
- The model never trains on an experience-evaluation portion before it is measured.
- The final temporal test is evaluated only after the stream is complete.
- CPU time and RAM are measured.
- Random seeds are fixed.

### EWC note

LightGBM is a tree model and does not naturally support parameter-level EWC in the same way as neural networks.

Therefore, to avoid inventing an invalid "EWC-LightGBM", this notebook implements EWC using a compact **MLP detector** as the EWC reference, while LightGBM remains the primary tree-based baseline.

This distinction is explicit in the results.


In [1]:
# 1. Install dependencies
!pip -q install datasets lightgbm psutil joblib scikit-learn torch

print("Dependencies installed.")


Dependencies installed.


In [2]:
# 2. Imports

import os
import gc
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import joblib

from datasets import load_dataset

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score
)
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE = Path("/content/carc_ids_phase2b")
RESULTS = BASE / "results"
ARTIFACTS = BASE / "artifacts"

RESULTS.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Working directory:", BASE)


Device: cpu
Working directory: /content/carc_ids_phase2b


# 3. Load the exact dataset

We use:

```python
lacg030175/UNSW-NB15
temporal
```

The final test set is kept independent.


In [3]:
# 3. Load dataset

ds = load_dataset(
    "lacg030175/UNSW-NB15",
    "temporal"
)

train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape)
print("Test :", test_df.shape)


README.md:   0%|          | 0.00/5.32k [00:00<?, ?B/s]

temporal/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.7MB            

temporal/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

temporal/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.25MB            

temporal/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/175341 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/82332 [00:00<?, ? examples/s]

Train: (175341, 44)
Test : (82332, 44)


# 4. Reconstruct the evidence-based continual-learning experiences

The grouping follows the corrected Phase-2A analysis.

We deliberately do not use all ten segments as separate learning tasks.

| Experience | Segments | Role |
|---|---|---|
| E1 | 3 | Initial attack regime |
| E2 | 4–6 | High-attack regime |
| E3 | 7 | Attack-family transition |
| E4 | 8 | Generic-dominant regime |
| E5 | 9–10 | Stable late regime |

Segments 1–2 are excluded from the supervised continual stream because they contain no attack examples and therefore cannot establish an attack detector.

They can still be used later for benign-only false-positive stress testing if required.


In [4]:
# 4. Create the same ten sequential segments

N_SEGMENTS = 10
indices = np.array_split(np.arange(len(train_df)), N_SEGMENTS)

segments = {
    sid: train_df.iloc[idx].copy()
    for sid, idx in enumerate(indices, start=1)
}

EXPERIENCES = {
    "E1_initial_attack": [3],
    "E2_high_attack": [4, 5, 6],
    "E3_attack_transition": [7],
    "E4_generic_dominant": [8],
    "E5_stable_late": [9, 10]
}

experience_dfs = {
    name: pd.concat([segments[s] for s in sids], ignore_index=True)
    for name, sids in EXPERIENCES.items()
}

exp_summary = []

for name, df in experience_dfs.items():
    exp_summary.append({
        "experience": name,
        "segments": ",".join(map(str, EXPERIENCES[name])),
        "rows": len(df),
        "benign": int((df.label == 0).sum()),
        "attack": int((df.label == 1).sum()),
        "attack_pct": float(df.label.mean()),
        "attack_categories": int(
            df.loc[df.label == 1, "attack_cat"].nunique()
        )
    })

exp_summary_df = pd.DataFrame(exp_summary)
display(exp_summary_df)

exp_summary_df.to_csv(
    RESULTS / "experience_summary.csv", index=False
)


,experience,segments,rows,benign,attack,attack_pct,attack_categories
0,E1_initial_attack,3,17534,12842,4692,0.267594,8
1,E2_high_attack,"4,5,6",52602,5405,47197,0.897247,8
2,E3_attack_transition,7,17534,2684,14850,0.846926,9
3,E4_generic_dominant,8,17534,0,17534,1.000000,9
4,E5_stable_late,"9,10",35068,0,35068,1.000000,9


# 5. Leakage-safe feature preparation

We remove:

- `label`;
- `attack_cat`;
- identifiers.

We do not use the behavioral features from Phase 1 because the Phase-1B experiment did not demonstrate a meaningful benefit.

The original network-flow feature representation is therefore the baseline representation.


In [5]:
# 5. Prepare raw features

DROP = [
    c for c in ["label", "attack_cat", "id", "ID", "index"]
    if c in train_df.columns
]

X_train_raw = train_df.drop(columns=DROP, errors="ignore").copy()
X_test_raw = test_df.drop(columns=DROP, errors="ignore").copy()

y_train = train_df["label"].astype(int).to_numpy()
y_test = test_df["label"].astype(int).to_numpy()

print("Dropped:", DROP)
print("Features:", X_train_raw.shape[1])


Dropped: ['label', 'attack_cat']
Features: 42


In [6]:
# 6. Fit one global preprocessing representation using TRAINING DATA ONLY

numeric_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X_train_raw.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_cols)
])

# Fit only on the supplied training split.
X_train_proc = preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_test_proc = preprocessor.transform(X_test_raw).astype(np.float32)

print("Processed train:", X_train_proc.shape)
print("Processed test :", X_test_proc.shape)

joblib.dump(preprocessor, ARTIFACTS / "phase2b_preprocessor.joblib")


Processed train: (175341, 194)
Processed test : (82332, 194)


['/content/carc_ids_phase2b/artifacts/phase2b_preprocessor.joblib']

# 7. Map each experience to processed arrays

Each experience is split into:

- **adaptation portion:** first 70% of that experience;
- **evaluation portion:** final 30%.

This preserves sequence order within each experience.

The evaluation portion is never used to update the model before its own evaluation.


In [7]:
# 7. Build experience arrays

experience_arrays = {}

for name, df in experience_dfs.items():
    # Recover original dataframe positions through a temporary index.
    # Because experience_dfs was concatenated in segment order, its rows are
    # transformed independently here using the same fitted preprocessor.
    X = df.drop(columns=DROP, errors="ignore").copy()
    y = df["label"].astype(int).to_numpy()

    Xp = preprocessor.transform(X).astype(np.float32)

    split = int(len(Xp) * 0.70)

    # Ensure both parts exist.
    split = max(1, min(split, len(Xp)-1))

    experience_arrays[name] = {
        "X_adapt": Xp[:split],
        "y_adapt": y[:split],
        "X_eval": Xp[split:],
        "y_eval": y[split:]
    }

    print(
        name,
        "adapt:", Xp[:split].shape,
        "eval:", Xp[split:].shape
    )


E1_initial_attack adapt: (12273, 194) eval: (5261, 194)
E2_high_attack adapt: (36821, 194) eval: (15781, 194)
E3_attack_transition adapt: (12273, 194) eval: (5261, 194)
E4_generic_dominant adapt: (12273, 194) eval: (5261, 194)
E5_stable_late adapt: (24547, 194) eval: (10521, 194)


# 8. Metrics

We evaluate every experience using:

- Accuracy
- Precision
- Recall
- F1
- FPR
- ROC-AUC
- PR-AUC

The most important continual-learning metrics are:

### New-regime performance

Performance on the current experience after adaptation.

### Retention

Performance on previously seen experiences after later updates.

### Forgetting

For each prior experience:

\[
F_i = \max_t M_{i,t} - M_{i,T}
\]

where `M` is F1 and `T` is the final experience.

We report both per-experience forgetting and mean forgetting.


In [8]:
# 8. Metric helpers

def binary_metrics(y_true, probs, threshold=0.5):
    pred = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0,1]
    ).ravel()

    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "roc_auc": roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, probs) if len(np.unique(y_true)) == 2 else np.nan,
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)
    }

def measure_resource():
    proc = psutil.Process(os.getpid())
    return {
        "rss_mb": proc.memory_info().rss / (1024**2),
        "cpu_percent": psutil.cpu_percent(interval=0.1)
    }


# 9. Model A — Static LightGBM

The static model is trained only once on E1 adaptation data.

It is then evaluated on every subsequent experience without further learning.

This establishes the amount of performance degradation caused by distribution change without adaptation.


In [9]:
# 9. Static LightGBM

e1 = experience_arrays["E1_initial_attack"]

static_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

resource_before = measure_resource()
t0 = time.perf_counter()

static_model.fit(e1["X_adapt"], e1["y_adapt"])

static_train_time = time.perf_counter() - t0
resource_after = measure_resource()

static_results = []

for exp_name, data in experience_arrays.items():
    probs = static_model.predict_proba(data["X_eval"])[:,1]

    m = binary_metrics(data["y_eval"], probs)
    m.update({
        "method": "Static_LightGBM",
        "experience": exp_name,
        "train_time_sec": static_train_time if exp_name == "E1_initial_attack" else 0.0,
        "rss_mb_after": resource_after["rss_mb"]
    })

    static_results.append(m)

static_results_df = pd.DataFrame(static_results)

display(
    static_results_df[
        ["method","experience","f1","recall","precision","fpr","roc_auc","pr_auc"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,method,experience,f1,recall,precision,fpr,roc_auc,pr_auc
0,Static_LightGBM,E1_initial_attack,0.0,0.0,0.0,0.0,0.5,0.8918
1,Static_LightGBM,E2_high_attack,0.0,0.0,0.0,0.0,0.5,0.8941
2,Static_LightGBM,E3_attack_transition,0.0,0.0,0.0,NaN,NaN,NaN
3,Static_LightGBM,E4_generic_dominant,0.0,0.0,0.0,NaN,NaN,NaN
4,Static_LightGBM,E5_stable_late,0.0,0.0,0.0,NaN,NaN,NaN


# 10. Model B — Naive Continual LightGBM

At every experience, the model is updated using only the current experience's adaptation data.

No replay memory is used.

No explicit forgetting protection is used.

This is a deliberately simple continual-learning baseline.


In [10]:
# 10. Naive continual LightGBM

naive_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

naive_results = []
naive_train_times = []

for i, (exp_name, data) in enumerate(experience_arrays.items(), start=1):
    resource_before = measure_resource()
    t0 = time.perf_counter()

    # For LightGBM, each fit replaces the current ensemble.
    # This is intentionally a naive "retrain on current experience" baseline.
    naive_model.fit(data["X_adapt"], data["y_adapt"])

    train_time = time.perf_counter() - t0
    naive_train_times.append(train_time)

    resource_after = measure_resource()

    # Evaluate on every experience observed so far.
    for eval_name, eval_data in list(experience_arrays.items())[:i]:
        probs = naive_model.predict_proba(eval_data["X_eval"])[:,1]
        m = binary_metrics(eval_data["y_eval"], probs)

        m.update({
            "method": "Naive_CL_LightGBM",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "train_time_sec": train_time,
            "rss_mb_after": resource_after["rss_mb"]
        })

        naive_results.append(m)

naive_results_df = pd.DataFrame(naive_results)

display(
    naive_results_df[
        ["method","after_experience","evaluated_experience","f1","recall","fpr"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,method,after_experience,evaluated_experience,f1,recall,fpr
0,Naive_CL_LightGBM,E1_initial_attack,E1_initial_attack,0.0000,0.0000,0.0000
1,Naive_CL_LightGBM,E2_high_attack,E1_initial_attack,0.9256,0.9469,0.8172
2,Naive_CL_LightGBM,E2_high_attack,E2_high_attack,0.9538,0.9794,0.6278
3,Naive_CL_LightGBM,E3_attack_transition,E1_initial_attack,0.9495,0.9729,0.6309
4,Naive_CL_LightGBM,E3_attack_transition,E2_high_attack,0.9420,0.9863,0.9096
5,Naive_CL_LightGBM,E3_attack_transition,E3_attack_transition,0.9962,0.9924,NaN
6,Naive_CL_LightGBM,E4_generic_dominant,E1_initial_attack,0.0000,0.0000,0.0000
7,Naive_CL_LightGBM,E4_generic_dominant,E2_high_attack,0.0000,0.0000,0.0000
8,Naive_CL_LightGBM,E4_generic_dominant,E3_attack_transition,0.0000,0.0000,NaN
9,Naive_CL_LightGBM,E4_generic_dominant,E4_generic_dominant,0.0000,0.0000,NaN


# 11. Model C — Replay LightGBM

A bounded replay buffer retains a small number of historical training samples.

At each new experience:

1. current adaptation data is collected;
2. replay samples from previous experiences are added;
3. the model is retrained on current + replay data;
4. the replay buffer is updated.

This is a standard continual-learning baseline.

Replay memory is deliberately bounded to make the resource cost measurable.


In [11]:
# 11. Replay continual learning

REPLAY_PER_EXPERIENCE = 2000

rng = np.random.default_rng(SEED)

replay_buffer_X = []
replay_buffer_y = []

replay_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

replay_results = []

for i, (exp_name, data) in enumerate(experience_arrays.items(), start=1):

    X_current = data["X_adapt"]
    y_current = data["y_adapt"]

    # Build bounded replay set.
    if replay_buffer_X:
        X_replay = np.concatenate(replay_buffer_X, axis=0)
        y_replay = np.concatenate(replay_buffer_y, axis=0)

        X_fit = np.concatenate([X_current, X_replay], axis=0)
        y_fit = np.concatenate([y_current, y_replay], axis=0)
    else:
        X_fit = X_current
        y_fit = y_current

    resource_before = measure_resource()
    t0 = time.perf_counter()

    replay_model.fit(X_fit, y_fit)

    train_time = time.perf_counter() - t0
    resource_after = measure_resource()

    # Evaluate on all seen experiences.
    for eval_name, eval_data in list(experience_arrays.items())[:i]:
        probs = replay_model.predict_proba(eval_data["X_eval"])[:,1]
        m = binary_metrics(eval_data["y_eval"], probs)

        m.update({
            "method": "Replay_LightGBM",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "train_time_sec": train_time,
            "replay_size": sum(len(x) for x in replay_buffer_X),
            "rss_mb_after": resource_after["rss_mb"]
        })

        replay_results.append(m)

    # Add bounded samples from current experience.
    n = min(REPLAY_PER_EXPERIENCE, len(X_current))
    chosen = rng.choice(len(X_current), size=n, replace=False)

    replay_buffer_X.append(X_current[chosen])
    replay_buffer_y.append(y_current[chosen])

replay_results_df = pd.DataFrame(replay_results)

display(
    replay_results_df[
        ["method","after_experience","evaluated_experience","f1","recall","fpr","replay_size"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,method,after_experience,evaluated_experience,f1,recall,fpr,replay_size
0,Replay_LightGBM,E1_initial_attack,E1_initial_attack,0.0000,0.0000,0.0000,0
1,Replay_LightGBM,E2_high_attack,E1_initial_attack,0.9221,0.9337,0.7540,2000
2,Replay_LightGBM,E2_high_attack,E2_high_attack,0.9660,0.9693,0.3166,2000
3,Replay_LightGBM,E3_attack_transition,E1_initial_attack,0.9195,0.8922,0.3989,4000
4,Replay_LightGBM,E3_attack_transition,E2_high_attack,0.9632,0.9551,0.2364,4000
5,Replay_LightGBM,E3_attack_transition,E3_attack_transition,0.9848,0.9700,NaN,4000
6,Replay_LightGBM,E4_generic_dominant,E1_initial_attack,0.9264,0.9162,0.5097,6000
7,Replay_LightGBM,E4_generic_dominant,E2_high_attack,0.9629,0.9609,0.2944,6000
8,Replay_LightGBM,E4_generic_dominant,E3_attack_transition,0.9906,0.9814,NaN,6000
9,Replay_LightGBM,E4_generic_dominant,E4_generic_dominant,0.9933,0.9867,NaN,6000


# 12. Model D — EWC Neural Continual Learner

EWC is implemented with a compact MLP because EWC requires parameter-level regularization.

This is not presented as an "EWC-LightGBM".

The MLP is kept intentionally small so that the experiment remains feasible in free Colab resources.

EWC objective:

\[
L = L_{current}
+
\frac{\lambda}{2}
\sum_i F_i(\theta_i-\theta_i^*)^2
\]

where:

- `F_i` is the estimated Fisher importance;
- `theta_i*` is the previous parameter value;
- `lambda` controls forgetting protection.


In [12]:
# 12. Compact MLP

class SmallMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        hidden = min(128, max(32, input_dim // 4))

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

def make_loader(X, y, batch_size=256, shuffle=True):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle
    )

def train_mlp_ewc(
    model,
    X,
    y,
    old_params=None,
    fisher=None,
    ewc_lambda=100.0,
    epochs=5,
    lr=1e-3
):
    model.train()

    loader = make_loader(X, y)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    start = time.perf_counter()

    for epoch in range(epochs):
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            if old_params is not None and fisher is not None:
                penalty = 0.0

                for name, param in model.named_parameters():
                    penalty = penalty + (
                        fisher[name] *
                        (param - old_params[name]) ** 2
                    ).sum()

                loss = loss + (ewc_lambda / 2.0) * penalty

            loss.backward()
            optimizer.step()

    return time.perf_counter() - start

@torch.no_grad()
def mlp_predict(model, X):
    model.eval()

    loader = make_loader(
        X,
        np.zeros(len(X), dtype=np.float32),
        shuffle=False
    )

    outputs = []

    for xb, _ in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        outputs.append(torch.sigmoid(logits).cpu().numpy())

    return np.concatenate(outputs)

def estimate_fisher(model, X, y):
    model.eval()

    loader = make_loader(X, y)
    criterion = nn.BCEWithLogitsLoss()

    fisher = {
        name: torch.zeros_like(param, device=DEVICE)
        for name, param in model.named_parameters()
    }

    count = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        model.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()

        batch_size = len(xb)
        count += batch_size

        for name, param in model.named_parameters():
            if param.grad is not None:
                fisher[name] += (
                    param.grad.detach() ** 2
                ) * batch_size

    for name in fisher:
        fisher[name] /= max(count, 1)

    return fisher


In [13]:
# 13. Run EWC experiment

# Input dimensionality is the processed feature dimension.
input_dim = X_train_proc.shape[1]

ewc_model = SmallMLP(input_dim).to(DEVICE)

ewc_lambda = 100.0
EWC_EPOCHS = 5

old_params = None
fisher = None

ewc_results = []

for i, (exp_name, data) in enumerate(experience_arrays.items(), start=1):

    resource_before = measure_resource()

    train_time = train_mlp_ewc(
        ewc_model,
        data["X_adapt"],
        data["y_adapt"],
        old_params=old_params,
        fisher=fisher,
        ewc_lambda=ewc_lambda,
        epochs=EWC_EPOCHS,
        lr=1e-3
    )

    resource_after = measure_resource()

    # Evaluate on all seen experiences.
    for eval_name, eval_data in list(experience_arrays.items())[:i]:

        probs = mlp_predict(
            ewc_model,
            eval_data["X_eval"]
        )

        m = binary_metrics(
            eval_data["y_eval"],
            probs
        )

        m.update({
            "method": "EWC_MLP",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "train_time_sec": train_time,
            "rss_mb_after": resource_after["rss_mb"]
        })

        ewc_results.append(m)

    # Estimate Fisher after learning the current experience.
    fisher = estimate_fisher(
        ewc_model,
        data["X_adapt"],
        data["y_adapt"]
    )

    old_params = {
        name: param.detach().clone()
        for name, param in ewc_model.named_parameters()
    }

ewc_results_df = pd.DataFrame(ewc_results)

display(
    ewc_results_df[
        ["method","after_experience","evaluated_experience","f1","recall","fpr"]
    ].round(4)
)


,method,after_experience,evaluated_experience,f1,recall,fpr
0,EWC_MLP,E1_initial_attack,E1_initial_attack,0.0000,0.0000,0.0000
1,EWC_MLP,E2_high_attack,E1_initial_attack,0.9358,0.9808,0.9508
2,EWC_MLP,E2_high_attack,E2_high_attack,0.9525,0.9908,0.7558
3,EWC_MLP,E3_attack_transition,E1_initial_attack,0.9453,1.0000,0.9543
4,EWC_MLP,E3_attack_transition,E2_high_attack,0.9458,1.0000,0.9683
5,EWC_MLP,E3_attack_transition,E3_attack_transition,1.0000,1.0000,NaN
6,EWC_MLP,E4_generic_dominant,E1_initial_attack,0.9453,1.0000,0.9543
7,EWC_MLP,E4_generic_dominant,E2_high_attack,0.9457,1.0000,0.9689
8,EWC_MLP,E4_generic_dominant,E3_attack_transition,1.0000,1.0000,NaN
9,EWC_MLP,E4_generic_dominant,E4_generic_dominant,1.0000,1.0000,NaN


# 13. Continual-learning forgetting analysis

For each method and each experience:

\[
Forgetting_i =
\max_{t\ge i} F1_{i,t}
-
F1_{i,final}
\]

A positive value means the model lost performance on that earlier experience after later learning.

Lower is better.

We also report final retention across all experiences.


In [14]:
# 14. Forgetting computation

def forgetting_table(results_df, method_name):
    rows = []

    if method_name == "Static_LightGBM":
        # Static results contain only one evaluation per experience.
        pivot = results_df.pivot(
            index="experience",
            columns="experience",
            values="f1"
        )
        return pd.DataFrame()

    # Continual methods
    for exp in results_df["evaluated_experience"].unique():
        g = results_df[
            results_df["evaluated_experience"] == exp
        ].copy()

        if g.empty:
            continue

        initial = g["f1"].max()
        final = g.sort_values(
            "after_experience"
        )["f1"].iloc[-1]

        rows.append({
            "method": method_name,
            "experience": exp,
            "max_f1_seen": initial,
            "final_f1": final,
            "forgetting": initial-final
        })

    return pd.DataFrame(rows)

forget_naive = forgetting_table(
    naive_results_df,
    "Naive_CL_LightGBM"
)

forget_replay = forgetting_table(
    replay_results_df,
    "Replay_LightGBM"
)

forget_ewc = forgetting_table(
    ewc_results_df,
    "EWC_MLP"
)

forgetting_all = pd.concat(
    [forget_naive, forget_replay, forget_ewc],
    ignore_index=True
)

display(forgetting_all.round(6))

forgetting_all.to_csv(
    RESULTS/"forgetting_analysis.csv",
    index=False
)


,method,experience,max_f1_seen,final_f1,forgetting
0,Naive_CL_LightGBM,E1_initial_attack,0.949459,0.000000,0.949459
1,Naive_CL_LightGBM,E2_high_attack,0.953794,0.000000,0.953794
2,Naive_CL_LightGBM,E3_attack_transition,0.996184,0.000000,0.996184
3,Naive_CL_LightGBM,E4_generic_dominant,0.000000,0.000000,0.000000
4,Naive_CL_LightGBM,E5_stable_late,0.000000,0.000000,0.000000
5,Replay_LightGBM,E1_initial_attack,0.926409,0.922193,0.004216
6,Replay_LightGBM,E2_high_attack,0.966026,0.959831,0.006195
7,Replay_LightGBM,E3_attack_transition,0.993303,0.993303,0.000000
8,Replay_LightGBM,E4_generic_dominant,0.993303,0.992821,0.000482
9,Replay_LightGBM,E5_stable_late,0.997523,0.997523,0.000000


# 15. Final stream performance comparison

We compare the final performance after all experiences have been processed.

For each method, we report:

- F1 on every previously observed experience;
- mean final F1;
- mean forgetting.

This is the core Phase-2B comparison.


In [15]:
# 15. Final retention table

final_rows = []

# Static
for exp_name, data in experience_arrays.items():
    row = static_results_df[
        static_results_df["experience"] == exp_name
    ].iloc[0]

    final_rows.append({
        "method": "Static_LightGBM",
        "experience": exp_name,
        "final_f1": row["f1"],
        "final_recall": row["recall"],
        "final_fpr": row["fpr"]
    })

# Continual methods
for method_name, df in [
    ("Naive_CL_LightGBM", naive_results_df),
    ("Replay_LightGBM", replay_results_df),
    ("EWC_MLP", ewc_results_df)
]:
    last_after = list(EXPERIENCES.keys())[-1]

    g = df[df["after_experience"] == last_after]

    for _, row in g.iterrows():
        final_rows.append({
            "method": method_name,
            "experience": row["evaluated_experience"],
            "final_f1": row["f1"],
            "final_recall": row["recall"],
            "final_fpr": row["fpr"]
        })

final_retention_df = pd.DataFrame(final_rows)

display(final_retention_df.round(4))

final_retention_df.to_csv(
    RESULTS/"final_stream_retention.csv",
    index=False
)


,method,experience,final_f1,final_recall,final_fpr
0,Static_LightGBM,E1_initial_attack,0.0000,0.0000,0.0000
1,Static_LightGBM,E2_high_attack,0.0000,0.0000,0.0000
2,Static_LightGBM,E3_attack_transition,0.0000,0.0000,NaN
3,Static_LightGBM,E4_generic_dominant,0.0000,0.0000,NaN
4,Static_LightGBM,E5_stable_late,0.0000,0.0000,NaN
5,Naive_CL_LightGBM,E1_initial_attack,0.0000,0.0000,0.0000
6,Naive_CL_LightGBM,E2_high_attack,0.0000,0.0000,0.0000
7,Naive_CL_LightGBM,E3_attack_transition,0.0000,0.0000,NaN
8,Naive_CL_LightGBM,E4_generic_dominant,0.0000,0.0000,NaN
9,Naive_CL_LightGBM,E5_stable_late,0.0000,0.0000,NaN


In [16]:
# 16. Aggregate continual-learning results

aggregate_rows = []

for method in final_retention_df["method"].unique():
    g = final_retention_df[
        final_retention_df["method"] == method
    ]

    aggregate_rows.append({
        "method": method,
        "mean_final_f1": g["final_f1"].mean(),
        "mean_final_recall": g["final_recall"].mean(),
        "mean_final_fpr": g["final_fpr"].mean()
    })

for method, g in forgetting_all.groupby("method"):
    row = next(
        r for r in aggregate_rows
        if r["method"] == method
    )
    row["mean_forgetting"] = g["forgetting"].mean()

aggregate_df = pd.DataFrame(aggregate_rows)

display(aggregate_df.round(6))

aggregate_df.to_csv(
    RESULTS/"continual_learning_aggregate_results.csv",
    index=False
)


,method,mean_final_f1,mean_final_recall,mean_final_fpr,mean_forgetting
0,Static_LightGBM,0.000000,0.000000,0.000000,NaN
1,Naive_CL_LightGBM,0.000000,0.000000,0.000000,0.579887
2,Replay_LightGBM,0.973134,0.971549,0.528285,0.002179
3,EWC_MLP,0.978171,1.000000,0.963389,0.001398


# 17. Adaptation-cost comparison

The final proposed controller will eventually need to optimize security gain against computational cost.

Phase 2B therefore records:

- training time;
- process RSS;
- replay memory size where applicable.

This is not yet the proposed utility function.

It is the evidence required to design it.


In [17]:
# 17. Resource summaries

resource_rows = []

for method, df in [
    ("Naive_CL_LightGBM", naive_results_df),
    ("Replay_LightGBM", replay_results_df),
    ("EWC_MLP", ewc_results_df)
]:
    resource_rows.append({
        "method": method,
        "total_training_time_sec": df.groupby(
            "after_experience"
        )["train_time_sec"].first().sum(),
        "max_rss_mb": df["rss_mb_after"].max(),
        "mean_rss_mb": df["rss_mb_after"].mean()
    })

resource_summary = pd.DataFrame(resource_rows)

resource_summary.loc[
    len(resource_summary)
] = [
    "Static_LightGBM",
    static_train_time,
    resource_after["rss_mb"],
    resource_after["rss_mb"]
]

display(resource_summary.round(4))

resource_summary.to_csv(
    RESULTS/"resource_summary.csv",
    index=False
)


,method,total_training_time_sec,max_rss_mb,mean_rss_mb
0,Naive_CL_LightGBM,17.7194,1292.4883,1292.4549
1,Replay_LightGBM,9.4469,1342.2227,1322.1753
2,EWC_MLP,21.4460,1426.0156,1424.3378
3,Static_LightGBM,0.7116,1426.0156,1426.0156


# 18. Final independent temporal test

The final test is evaluated only after the continual-learning stream.

For LightGBM methods, the final model is the model after E5.

For EWC, the final MLP is evaluated separately.

This is not used to tune any hyperparameter.

The purpose is to determine whether the continual-learning process generalizes to the held-out temporal test.


In [18]:
# 18. Final temporal test evaluation

# Static
static_test_probs = static_model.predict_proba(X_test_proc)[:,1]
static_test_metrics = binary_metrics(y_test, static_test_probs)

# Naive final model
naive_test_probs = naive_model.predict_proba(X_test_proc)[:,1]
naive_test_metrics = binary_metrics(y_test, naive_test_probs)

# Replay final model
replay_test_probs = replay_model.predict_proba(X_test_proc)[:,1]
replay_test_metrics = binary_metrics(y_test, replay_test_probs)

# EWC final model
ewc_test_probs = mlp_predict(ewc_model, X_test_proc)
ewc_test_metrics = binary_metrics(y_test, ewc_test_probs)

final_test_rows = []

for name, metrics in [
    ("Static_LightGBM", static_test_metrics),
    ("Naive_CL_LightGBM", naive_test_metrics),
    ("Replay_LightGBM", replay_test_metrics),
    ("EWC_MLP", ewc_test_metrics)
]:
    row = {"method": name}
    row.update(metrics)
    final_test_rows.append(row)

final_test_df = pd.DataFrame(final_test_rows)

display(
    final_test_df[
        ["method","accuracy","precision","recall","f1","fpr","roc_auc","pr_auc"]
    ].round(4)
)

final_test_df.to_csv(
    RESULTS/"final_temporal_test_comparison.csv",
    index=False
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,method,accuracy,precision,recall,f1,fpr,roc_auc,pr_auc
0,Static_LightGBM,0.4494,0.0000,0.0000,0.0000,0.0000,0.5000,0.5506
1,Naive_CL_LightGBM,0.4494,0.0000,0.0000,0.0000,0.0000,0.5000,0.5506
2,Replay_LightGBM,0.8549,0.7999,0.9821,0.8817,0.3010,0.9766,0.9825
3,EWC_MLP,0.5611,0.5564,1.0000,0.7150,0.9766,0.6950,0.6682


# 19. Interpretation rules

Do not claim that a method is better merely because its final test F1 is higher.

A useful continual-learning method should ideally show:

1. competitive current-regime detection;
2. lower forgetting on previous regimes;
3. acceptable false-positive behavior;
4. reasonable adaptation time/memory;
5. no dependence on unrealistic unlimited replay memory.

The proposed resource-aware controller will only be justified if the benchmark shows a real trade-off worth optimizing.

### Important

This notebook is a benchmark, not yet the final proposed system.

If naive continual learning performs no better than static learning, that is an important result.

If replay provides most of the benefit at low cost, the proposed controller must beat or improve that trade-off to be worth claiming novelty.


In [19]:
# 20. Save all Phase-2B artifacts

joblib.dump(static_model, ARTIFACTS/"static_lightgbm.joblib")
joblib.dump(naive_model, ARTIFACTS/"naive_continual_lightgbm.joblib")
joblib.dump(replay_model, ARTIFACTS/"replay_lightgbm.joblib")
torch.save(ewc_model.state_dict(), ARTIFACTS/"ewc_mlp_state.pt")

protocol = {
    "dataset": "lacg030175/UNSW-NB15",
    "config": "temporal",
    "experiences": EXPERIENCES,
    "experience_split": "first 70 percent adaptation, final 30 percent evaluation",
    "final_test_used_for_stream_training": False,
    "models": [
        "Static LightGBM",
        "Naive continual LightGBM",
        "Replay LightGBM",
        "EWC MLP"
    ],
    "replay_per_experience": REPLAY_PER_EXPERIENCE,
    "ewc_lambda": ewc_lambda,
    "ewc_epochs": EWC_EPOCHS,
    "seed": SEED,
    "note": (
        "EWC is implemented with a compact MLP because parameter-level EWC "
        "is not naturally defined for LightGBM."
    )
}

with open(RESULTS/"phase2b_protocol.json","w") as f:
    json.dump(protocol, f, indent=2)

bundle = shutil.make_archive(
    str(BASE/"phase2b_artifacts"),
    "zip",
    root_dir=BASE
)

print("Created:", bundle)
print("\nImportant outputs:")
for p in sorted(RESULTS.glob("*")):
    print(" -", p)


Created: /content/carc_ids_phase2b/phase2b_artifacts.zip

Important outputs:
 - /content/carc_ids_phase2b/results/continual_learning_aggregate_results.csv
 - /content/carc_ids_phase2b/results/experience_summary.csv
 - /content/carc_ids_phase2b/results/final_stream_retention.csv
 - /content/carc_ids_phase2b/results/final_temporal_test_comparison.csv
 - /content/carc_ids_phase2b/results/forgetting_analysis.csv
 - /content/carc_ids_phase2b/results/phase2b_protocol.json
 - /content/carc_ids_phase2b/results/resource_summary.csv


# 21. What to send back

Send these outputs:

1. `experience_summary.csv`
2. `final_stream_retention.csv`
3. `continual_learning_aggregate_results.csv`
4. `forgetting_analysis.csv`
5. `resource_summary.csv`
6. `final_temporal_test_comparison.csv`

### Decision after this run

We will determine whether:

- continual learning actually helps;
- replay is sufficient;
- forgetting is substantial;
- adaptation cost is material;
- the resource-aware selective controller is justified.

Only then will we implement:

**Phase 2C — Resource-Aware Selective Adaptation Utility**

followed by the IPS response layer and, last, the local LLM.
